# VPU 硬件测试 - 分步测试流程

本notebook将VPU测试分为以下步骤：
1. **BRAM读写测试** - 测试global_bram和inst_bram基本读写
2. **VPU寄存器测试** - 测试VPU_AXI_Regs寄存器访问
3. **INST_Decoder测试** - 测试指令解码器基本功能
4. **CDMA搬运测试** - 通过指令解码器控制CDMA
5. **VPU运算测试** - 测试VPU Max Pooling功能

## 地址映射
- `0x1000_0000` - global_bram (1MB) - 数据暂存区
- `0x1020_0000` - inst_bram (128KB) - 指令存储区 [优化: 容量减小以改善时序]  
- `0x1040_0000` - VPU GB (128KB) - VPU全局缓冲区
- `0x1042_0000` - VPU WB (128KB) - VPU权重缓冲区
- `0x1044_0000` - VPU_AXI_Regs (4KB) - VPU控制寄存器

In [ ]:
# 导入依赖库
import sys
import time
import struct
import numpy as np
from pathlib import Path

# 添加测试模块路径
parent_dir = Path.cwd()
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

from xdma_helpers import write_blob, read_blob, write_reg, read_reg
from xdma_helpers import GLOBAL_BRAM_BASE, INST_BRAM_BASE, VPU_GB_BASE, VPU_WB_BASE, VPU_REGS_BASE

print("✓ 模块导入成功")
print(f"地址映射:")
print(f"  GLOBAL_BRAM: 0x{GLOBAL_BRAM_BASE:08X}")
print(f"  INST_BRAM:   0x{INST_BRAM_BASE:08X}")
print(f"  VPU_GB:      0x{VPU_GB_BASE:08X}")
print(f"  VPU_WB:      0x{VPU_WB_BASE:08X}")
print(f"  VPU_REGS:    0x{VPU_REGS_BASE:08X}")

## 步骤 1: BRAM 基本读写测试

测试 global_bram 和 inst_bram 的基本读写功能

In [ ]:
# 测试 1.1: global_bram 读写 - 详细诊断版本
print("=" * 60)
print("测试 1.1: global_bram 读写 (256个fp32) - 详细诊断")
print("=" * 60)

# 生成256个fp32测试数据
num_elements = 256
test_data = np.arange(num_elements, dtype=np.float32)
print(f"写入数据大小: {len(test_data)} 个 fp32 ({len(test_data)*4} 字节)")
print(f"写入数据前8个: {test_data[:8]}")
print(f"写入数据后8个: {test_data[-8:]}")

# 写入数据
write_blob(GLOBAL_BRAM_BASE, test_data.tobytes())
print(f"✓ 数据已写入 0x{GLOBAL_BRAM_BASE:08X}")

# 读回数据
read_back = read_blob(GLOBAL_BRAM_BASE, len(test_data) * 4)
read_data = np.frombuffer(read_back, dtype=np.float32)

# 详细比较
print(f"\n【详细诊断】")
if np.array_equal(test_data, read_data):
    print(f"✓ global_bram 读写一致")
    print(f"  回读数据前8个: {read_data[:8]}")
    print(f"  回读数据后8个: {read_data[-8:]}")
else:
    print(f"✗ global_bram 读写不一致")
    print(f"  期望前8个: {test_data[:8]}")
    print(f"  实际前8个: {read_data[:8]}")
    
    # 找出所有不一致的位置
    diff_idx = np.where(test_data != read_data)[0]
    print(f"\n不一致统计:")
    print(f"  总数量: {len(diff_idx)}/{len(test_data)} ({100*len(diff_idx)/len(test_data):.1f}%)")
    
    if len(diff_idx) > 0:
        # 显示前10个不一致的位置
        print(f"\n前10个不一致位置:")
        for i in range(min(10, len(diff_idx))):
            idx = diff_idx[i]
            exp_val = test_data[idx]
            act_val = read_data[idx]
            exp_hex = struct.pack('f', exp_val).hex()
            act_hex = struct.pack('f', act_val).hex()
            print(f"  [{idx:3d}] 期望={exp_val:8.2f} (0x{exp_hex}), 实际={act_val:8.2f} (0x{act_hex})")
        
        # 检查是否有规律（例如每隔N个字节出错）
        if len(diff_idx) > 1:
            diffs = np.diff(diff_idx)
            unique_diffs = np.unique(diffs)
            print(f"\n不一致位置的间隔: {unique_diffs[:10]}")
            if len(unique_diffs) == 1:
                print(f"  → 固定间隔: 每隔 {unique_diffs[0]} 个元素")
        
        # 字节级分析：检查是否某个字节位置总是出错
        print(f"\n字节级分析 (前3个错误元素):")
        for i in range(min(3, len(diff_idx))):
            idx = diff_idx[i]
            exp_bytes = struct.pack('f', test_data[idx])
            act_bytes = struct.pack('f', read_data[idx])
            print(f"  元素[{idx}]:")
            print(f"    期望字节: {exp_bytes.hex()} = [{' '.join(f'{b:02x}' for b in exp_bytes)}]")
            print(f"    实际字节: {act_bytes.hex()} = [{' '.join(f'{b:02x}' for b in act_bytes)}]")
            for j in range(4):
                if exp_bytes[j] != act_bytes[j]:
                    print(f"      → 字节{j}不同: {exp_bytes[j]:02x} vs {act_bytes[j]:02x}")

In [ ]:
# 测试 1.2: inst_bram 读写 (fp32)
print("=" * 60)
print("测试 1.2: inst_bram 读写 (256个fp32)")
print("=" * 60)

# 生成256个fp32测试数据
test_data = np.arange(256, dtype=np.float32)
print(f"写入数据大小: {len(test_data)} 个 fp32 ({len(test_data)*4} 字节)")
print(f"写入数据前8个: {test_data[:8]}")

write_blob(INST_BRAM_BASE, test_data.tobytes())
read_back = read_blob(INST_BRAM_BASE, len(test_data) * 4)
read_data = np.frombuffer(read_back, dtype=np.float32)

if np.array_equal(test_data, read_data):
    print(f"✓ inst_bram 读写一致")
    print(f"  回读数据前8个: {read_data[:8]}")
    print(f"  回读数据后8个: {read_data[-8:]}")
else:
    print(f"✗ inst_bram 读写失败")
    print(f"  期望前8个: {test_data[:8]}")
    print(f"  实际前8个: {read_data[:8]}")
    # 找出不一致的位置
    diff_idx = np.where(test_data != read_data)[0]
    if len(diff_idx) > 0:
        print(f"  不一致数量: {len(diff_idx)}/{len(test_data)}")
        print(f"  首个不一致位置: {diff_idx[0]}, 期望={test_data[diff_idx[0]]}, 实际={read_data[diff_idx[0]]}")

## 步骤 2: VPU 寄存器测试

测试 VPU_AXI_Regs 寄存器的读写功能

In [ ]:
# 定义VPU寄存器偏移（新架构）
# 可读写的寄存器：
REG_STATUS = 0x04          # [0] VPU ready (只读)
REG_DECODER_CTRL = 0x38    # [0] Decoder start (读写)
REG_INST_COUNT = 0x3C      # 指令数量 (读写)
REG_DECODER_STATUS = 0x40  # [0] busy, [1] done, [31] error (只读)

# 已废弃的寄存器（读取会返回0）：
# 0x08-0x34: UNIT_CHOOSE, mp_src_h, mp_src_w等VPU配置寄存器
# 这些现在由INST_Decoder的VPU_EXEC指令直接控制，不经过AXI寄存器

print("=" * 60)
print("测试 2: VPU 寄存器读写")
print("=" * 60)

# 读取VPU状态
status = read_reg(VPU_REGS_BASE, REG_STATUS)
print(f"VPU Status = 0x{status:08X}")
print(f"  [0] ready = {status & 0x1}")

# 读取解码器状态
decoder_status = read_reg(VPU_REGS_BASE, REG_DECODER_STATUS)
print(f"Decoder Status = 0x{decoder_status:08X}")
print(f"  [0] busy = {decoder_status & 0x1}")
print(f"  [1] done = {(decoder_status >> 1) & 0x1}")

# 测试可读写的INST_COUNT寄存器
print(f"\n测试解码器控制寄存器...")
write_reg(VPU_REGS_BASE, REG_INST_COUNT, 0x12345678)
inst_count_rb = read_reg(VPU_REGS_BASE, REG_INST_COUNT)
if inst_count_rb == 0x12345678:
    print(f"✓ INST_COUNT 寄存器读写成功 (0x{inst_count_rb:08X})")
else:
    print(f"✗ INST_COUNT 读写失败, 期望 0x12345678, 实际 0x{inst_count_rb:08X}")

# 架构说明
print(f"\n架构说明:")
print(f"  【新架构】软件 -> INST_Decoder -> VPU")
print(f"  - 软件只写 DECODER_CTRL/INST_COUNT 启动指令序列")
print(f"  - INST_Decoder 解析指令，直接控制 VPU 和 CDMA")
print(f"  - VPU配置寄存器(0x08-0x34)已从AXI寄存器接口移除")
print(f"  - 读取废弃地址会返回 0x00000000")
print(f"✓ 寄存器接口工作正常")

## 步骤 3: 指令解码器功能测试

定义指令编码函数和解码器控制函数

In [ ]:
# 指令操作码定义
OP_NOP = 0x0
OP_CDMA_COPY = 0x1
OP_VPU_EXEC = 0x2
OP_WAIT_CDMA = 0x3
OP_WAIT_VPU = 0x4
OP_END = 0xF

# VPU 单元代码定义 (来自 Global_VPU.v)
UNIT_DQA = 1   # DeQuantize + Activation
UNIT_NN  = 2   # Neural Network LUT
UNIT_QA  = 3   # Quantize + Activation
UNIT_MP  = 4   # Max Pooling
UNIT_US  = 5   # UpSample
UNIT_AD  = 6   # Add (元素级加法)

print("VPU 单元代码:")
print(f"  UNIT_DQA = {UNIT_DQA} (DeQuantize)")
print(f"  UNIT_NN  = {UNIT_NN} (NN LUT)")
print(f"  UNIT_QA  = {UNIT_QA} (Quantize)")
print(f"  UNIT_MP  = {UNIT_MP} (Max Pooling)")
print(f"  UNIT_US  = {UNIT_US} (UpSample)")
print(f"  UNIT_AD  = {UNIT_AD} (Add)")

def _make_header(opcode, body_length, flags=0):
    """构造指令头: [31:28]OPCODE | [27:24]FLAGS | [23:0]BODY_LENGTH"""
    return ((opcode & 0xF) << 28) | ((flags & 0xF) << 24) | (body_length & 0xFFFFFF)

def encode_cdma_copy(src_addr, dst_addr, length):
    """CDMA_COPY指令: 4 words"""
    header = _make_header(OP_CDMA_COPY, 12)
    return struct.pack('<IIII', header, src_addr, dst_addr, length)

def encode_wait_cdma():
    """WAIT_CDMA指令: 1 word"""
    header = _make_header(OP_WAIT_CDMA, 0)
    return struct.pack('<I', header)

def encode_wait_vpu():
    """WAIT_VPU指令: 1 word"""
    header = _make_header(OP_WAIT_VPU, 0)
    return struct.pack('<I', header)

def encode_vpu_exec(unit_choose, src_addr, src2_addr, src_c, src_h, src_w,
                    bias_addr, scale_addr, dst_addr, addr_break, addr_s, addr_t):
    """VPU_EXEC指令: 13 words"""
    header = _make_header(OP_VPU_EXEC, 48)
    return struct.pack('<IIIIIIIIIIIII', header, unit_choose, src_addr, src2_addr,
                       src_c, src_h, src_w, bias_addr, scale_addr, dst_addr,
                       addr_break, addr_s, addr_t)

def encode_end():
    """END指令: 1 word"""
    header = _make_header(OP_END, 0)
    return struct.pack('<I', header)

def build_instruction_sequence(instructions):
    """合并指令列表"""
    return b''.join(instructions)

print("\n✓ 指令编码函数定义完成")

In [ ]:
# 解码器控制函数
def decoder_start(inst_count):
    """启动解码器"""
    write_reg(VPU_REGS_BASE, REG_DECODER_CTRL, 0x00)
    time.sleep(0.001)
    write_reg(VPU_REGS_BASE, REG_INST_COUNT, inst_count)
    write_reg(VPU_REGS_BASE, REG_DECODER_CTRL, 0x01)

def decoder_wait(timeout=5.0):
    """等待解码器完成"""
    deadline = time.time() + timeout
    seen_busy = False
    
    while time.time() < deadline:
        status = read_reg(VPU_REGS_BASE, REG_DECODER_STATUS)
        busy = status & 0x01
        done = (status >> 1) & 0x01
        error = (status >> 31) & 0x01
        
        if error:
            raise RuntimeError(f"Decoder error: status=0x{status:08X}")
        
        if busy:
            seen_busy = True
        
        if done and not busy:
            return status
        if seen_busy and status == 0:
            return status
        
        time.sleep(0.001)
    
    final_status = read_reg(VPU_REGS_BASE, REG_DECODER_STATUS)
    raise TimeoutError(f"Decoder timeout: status=0x{final_status:08X}")

print("✓ 解码器控制函数定义完成")

## 步骤 4: CDMA 搬运测试

通过指令解码器控制CDMA进行数据搬运

In [ ]:
# 测试 4.1: CDMA 搬运 global_bram → VPU_GB (多位置多数据测试)
print("=" * 60)
print("测试 4.1: CDMA 搬运 (global_bram → VPU_GB) - 多位置多数据")
print("=" * 60)

# 定义多个测试用例，每个用例使用不同的位置和数据
test_cases = [
    {
        'name': '测试1: 低地址 uint8',
        'src_offset': 0x0000,
        'dst_offset': 0x0000,
        'size': 512,
        'dtype': np.uint8,
        'data_gen': lambda n: np.arange(n, dtype=np.uint8)
    },
    {
        'name': '测试2: 中地址 fp32',
        'src_offset': 0x2000,
        'dst_offset': 0x1000,
        'size': 256,  # 256字节 = 64个fp32
        'dtype': np.float32,
        'data_gen': lambda n: np.linspace(0, 100, n, dtype=np.float32)
    },
    {
        'name': '测试3: 高地址 int16',
        'src_offset': 0x8000,
        'dst_offset': 0x4000,
        'size': 1024,  # 1024字节 = 512个int16
        'dtype': np.int16,
        'data_gen': lambda n: (np.random.rand(n) * 1000 - 500).astype(np.int16)
    }
]

# 执行所有测试用例
for i, test in enumerate(test_cases, 1):
    print(f"\n{'='*50}")
    print(f"{test['name']}")
    print(f"{'='*50}")
    
    # 计算元素数量
    elem_size = np.dtype(test['dtype']).itemsize
    num_elements = test['size'] // elem_size
    
    # 生成测试数据
    np.random.seed(42 + i)  # 固定种子保证可重复
    test_data_raw = test['data_gen'](num_elements)
    test_data_bytes = test_data_raw.tobytes()
    
    print(f"源地址: global_bram+0x{test['src_offset']:04X} (0x{GLOBAL_BRAM_BASE+test['src_offset']:08X})")
    print(f"目标地址: VPU_GB+0x{test['dst_offset']:04X} (0x{VPU_GB_BASE+test['dst_offset']:08X})")
    print(f"数据大小: {test['size']} 字节 = {num_elements} 个 {test['dtype']}")
    print(f"数据前4个: {test_data_raw[:4]}")
    
    # 写入测试数据到 global_bram
    write_blob(GLOBAL_BRAM_BASE + test['src_offset'], test_data_bytes)
    print(f"✓ 写入数据到 global_bram")
    
    # 构建CDMA指令
    instructions = [
        encode_cdma_copy(
            GLOBAL_BRAM_BASE + test['src_offset'], 
            VPU_GB_BASE + test['dst_offset'], 
            test['size']
        ),
        encode_wait_cdma(),
        encode_end(),
    ]
    inst_bytes = build_instruction_sequence(instructions)
    inst_count = len(inst_bytes) // 4
    
    # 写入指令并启动解码器
    write_blob(INST_BRAM_BASE, inst_bytes)
    
    start_time = time.time()
    decoder_start(inst_count)
    status = decoder_wait(timeout=3.0)
    elapsed = time.time() - start_time
    
    print(f"✓ CDMA完成: status=0x{status:08X}, 耗时 {elapsed:.3f}s")
    
    # 验证结果
    result_bytes = read_blob(VPU_GB_BASE + test['dst_offset'], test['size'])
    result_data = np.frombuffer(result_bytes, dtype=test['dtype'])
    
    if np.array_equal(test_data_raw, result_data):
        print(f"✓ {test['name']} 搬运成功")
        print(f"  VPU_GB 前4个: {result_data[:4]}")
        print(f"  VPU_GB 后4个: {result_data[-4:]}")
    else:
        print(f"✗ {test['name']} 搬运失败")
        # 找出不一致的位置
        diff_mask = test_data_raw != result_data
        diff_idx = np.where(diff_mask)[0]
        print(f"  不一致数量: {len(diff_idx)}/{len(test_data_raw)}")
        
        if len(diff_idx) > 0:
            # 显示前几个不一致的位置（最多显示5个）
            show_count = min(5, len(diff_idx))
            print(f"\n  前 {show_count} 个不一致位置:")
            
            for j in range(show_count):
                idx = diff_idx[j]
                # 显示不一致位置前后各3个元素的上下文
                start_idx = max(0, idx - 3)
                end_idx = min(len(test_data_raw), idx + 4)
                
                print(f"\n    [{j+1}] index={idx}, 字节偏移=0x{idx*elem_size:04X}")
                print(f"        期望值: {test_data_raw[idx]}")
                print(f"        实际值: {result_data[idx]}")
                print(f"        上下文[{start_idx}:{end_idx}]:")
                print(f"          期望: {test_data_raw[start_idx:end_idx]}")
                print(f"          实际: {result_data[start_idx:end_idx]}")
                # 用箭头标记不一致的位置
                marker_pos = idx - start_idx
                print(f"          指示: {' '*marker_pos}^^^^")
            
            if len(diff_idx) > show_count:
                print(f"\n  ... 还有 {len(diff_idx)-show_count} 个不一致位置未显示")
            
            # 分析不一致位置的分布
            if len(diff_idx) > 1:
                intervals = np.diff(diff_idx)
                print(f"\n  不一致间隔统计: min={intervals.min()}, max={intervals.max()}, mean={intervals.mean():.1f}")

print(f"\n{'='*60}")
print(f"测试 4.1 完成 - 共 {len(test_cases)} 个用例")
print(f"{'='*60}")

In [ ]:
# 测试 4.2: CDMA 往返搬运 (global_bram → VPU_GB → global_bram) - fp32
print("=" * 60)
print("测试 4.2: CDMA 往返搬运 (256个fp32)")
print("=" * 60)

num_elements = 256
size = num_elements * 4  # 256个fp32 = 1024字节
src_off = 0x2000
dst_off = 0x3000
test_data = np.arange(num_elements, dtype=np.float32)

print(f"数据大小: {num_elements} 个 fp32 ({size} 字节)")
print(f"测试数据前8个: {test_data[:8]}")

# 准备数据
write_blob(GLOBAL_BRAM_BASE + src_off, test_data.tobytes())
# 用特殊值填充目标区域以验证确实被覆盖
clear_data = np.full(num_elements, -999.0, dtype=np.float32)
write_blob(GLOBAL_BRAM_BASE + dst_off, clear_data.tobytes())
print(f"✓ 源数据写入 0x{GLOBAL_BRAM_BASE+src_off:08X}")
print(f"✓ 目标区域清空 0x{GLOBAL_BRAM_BASE+dst_off:08X} (填充-999.0)")

# 构建往返指令
instructions = [
    encode_cdma_copy(GLOBAL_BRAM_BASE + src_off, VPU_GB_BASE, size),
    encode_wait_cdma(),
    encode_cdma_copy(VPU_GB_BASE, GLOBAL_BRAM_BASE + dst_off, size),
    encode_wait_cdma(),
    encode_end(),
]
inst_bytes = build_instruction_sequence(instructions)
inst_count = len(inst_bytes) // 4

print(f"\n指令序列: {inst_count} words")
print(f"Step 1: 0x{GLOBAL_BRAM_BASE+src_off:08X} → 0x{VPU_GB_BASE:08X} ({size}B)")
print(f"Step 2: 0x{VPU_GB_BASE:08X} → 0x{GLOBAL_BRAM_BASE+dst_off:08X} ({size}B)")

write_blob(INST_BRAM_BASE, inst_bytes)
start_time = time.time()
decoder_start(inst_count)
status = decoder_wait(timeout=5.0)
elapsed = time.time() - start_time

print(f"✓ 往返搬运完成: status=0x{status:08X}, 耗时 {elapsed:.3f}s")

# 验证结果
result = np.frombuffer(read_blob(GLOBAL_BRAM_BASE + dst_off, size), dtype=np.float32)
if np.array_equal(test_data, result):
    print(f"✓ 往返搬运成功, 数据一致")
    print(f"  结果前8个: {result[:8]}")
    print(f"  结果后8个: {result[-8:]}")
else:
    print(f"✗ 往返搬运失败")
    print(f"  期望前8个: {test_data[:8]}")
    print(f"  实际前8个: {result[:8]}")
    # 找出不一致的位置
    diff_idx = np.where(test_data != result)[0]
    if len(diff_idx) > 0:
        print(f"  不一致数量: {len(diff_idx)}/{len(test_data)}")
        print(f"  首个不一致位置: {diff_idx[0]}, 期望={test_data[diff_idx[0]]}, 实际={result[diff_idx[0]]}")

## 步骤 5: VPU Max Pooling 运算测试

测试VPU的Max Pooling功能（完整流程）

In [40]:
# 测试 5: VPU Max Pooling 完整流程
print("=" * 60)
print("测试 5: VPU Max Pooling 运算")
print("=" * 60)

# Max Pooling 参数（匹配 mp_unit_fixed.sv）
ACT_CHANNEL_NUM = 128  # 通道数（硬编码为128）
ACT_HEIGHT = 10        # 输入高度
ACT_WIDTH = 10         # 输入宽度
KERNEL_SIZE = 5        # 5x5 pooling
PAD = 2                # padding=2 (SAME padding)

# 输出尺寸（SAME padding: 输出=输入）
OUT_HEIGHT = ACT_HEIGHT   # 10
OUT_WIDTH = ACT_WIDTH     # 10

# 数据大小（FP32 = 4 bytes）
ACT_SIZE = ACT_CHANNEL_NUM * ACT_HEIGHT * ACT_WIDTH  # 12800 个 FP32
ACT_BYTES = ACT_SIZE * 4  # 51200 bytes

OUT_SIZE = ACT_CHANNEL_NUM * OUT_HEIGHT * OUT_WIDTH  # 12800 个 FP32
OUT_BYTES = OUT_SIZE * 4  # 51200 bytes

# VPU 参数 - ★★★ 简化配置：输入输出都在地址 0 ★★★
UNIT_MP = 4
SRC_ADDR = 0x0
DST_ADDR = 0x0  # 输出覆盖输入（SAME padding，尺寸相同）

# mp_unit_fixed 不使用这些参数（已硬编码），但仍需传入
SRC_C = ACT_CHANNEL_NUM
SRC_H = OUT_HEIGHT
SRC_W = OUT_WIDTH

print(f"Max Pooling 参数 (FP32, 硬编码, 简化配置):")
print(f"  输入: C={ACT_CHANNEL_NUM}, H={ACT_HEIGHT}, W={ACT_WIDTH}")
print(f"  输出: C={SRC_C}, H={SRC_H}, W={SRC_W} (SAME padding)")
print(f"  SRC_ADDR = 0x{SRC_ADDR:X}")
print(f"  DST_ADDR = 0x{DST_ADDR:X} (覆盖输入)")
print(f"  数据大小: {ACT_BYTES} bytes")
print(f"  数据类型: FP32 (float32)")
print(f"  Kernel: {KERNEL_SIZE}x{KERNEL_SIZE}, Pad: {PAD}")

测试 5: VPU Max Pooling 运算
Max Pooling 参数 (FP32, 硬编码, 简化配置):
  输入: C=128, H=10, W=10
  输出: C=128, H=10, W=10 (SAME padding)
  SRC_ADDR = 0x0
  DST_ADDR = 0x0 (覆盖输入)
  数据大小: 51200 bytes
  数据类型: FP32 (float32)
  Kernel: 5x5, Pad: 2


In [41]:
# 准备测试数据 (FP32, HWC 格式)
print("\n准备输入数据 (FP32, HWC 格式)...")

# 生成测试数据: CHW 格式 (C, H, W)
shape_chw = (ACT_CHANNEL_NUM, ACT_HEIGHT, ACT_WIDTH)
act_data_chw = np.arange(np.prod(shape_chw), dtype=np.float32).reshape(shape_chw)

# ★★★ 关键：转换为 HWC 格式 (H, W, C) ★★★
# RTL 期望 HWC 布局：地址 = (h * W + w) * C + c
act_data = np.transpose(act_data_chw, (1, 2, 0))  # CHW -> HWC

# 转换为字节
act_data_bytes = act_data.tobytes()

print(f"✓ 生成测试数据: {len(act_data_bytes)} bytes")
print(f"  原始形状 (CHW): {shape_chw}")
print(f"  转换后形状 (HWC): {act_data.shape}")
print(f"  数据类型: {act_data.dtype}")
print(f"  数据范围: [{act_data.min():.1f}, {act_data.max():.1f}]")
print(f"  前8个值 (HWC): {act_data.flatten()[:8]}")
print(f"  对应原始 CHW 索引: channel=0, h=0, w=0..7")

# 预先将 dst_addr 对应范围清零，再写入输入并执行 Max Pooling 测试
dst_zeros = np.zeros(OUT_SIZE, dtype=np.float32)
dst_zero_bytes = dst_zeros.tobytes()
write_blob(VPU_GB_BASE + DST_ADDR, dst_zero_bytes)
write_blob(GLOBAL_BRAM_BASE + DST_ADDR, dst_zero_bytes)
print(f"\n✓ DST 区域已预写 0")
print(f"  VPU GB dst: 0x{VPU_GB_BASE+DST_ADDR:08X} (dst_addr=0x{DST_ADDR:X}, {OUT_BYTES} bytes)")
print(f"  global_bram: 0x{GLOBAL_BRAM_BASE+DST_ADDR:08X} ({OUT_BYTES} bytes)")

# 写入数据到 global_bram（与 DST 同址时覆盖上面的清零）
write_blob(GLOBAL_BRAM_BASE + SRC_ADDR, act_data_bytes)
print(f"✓ 数据已写入 global_bram (0x{GLOBAL_BRAM_BASE+SRC_ADDR:08X})")


准备输入数据 (FP32, HWC 格式)...
✓ 生成测试数据: 51200 bytes
  原始形状 (CHW): (128, 10, 10)
  转换后形状 (HWC): (10, 10, 128)
  数据类型: float32
  数据范围: [0.0, 12799.0]
  前8个值 (HWC): [  0. 100. 200. 300. 400. 500. 600. 700.]
  对应原始 CHW 索引: channel=0, h=0, w=0..7

✓ DST 区域已预写 0
  VPU GB dst: 0x10400000 (dst_addr=0x0, 51200 bytes)
  global_bram: 0x10000000 (51200 bytes)
✓ 数据已写入 global_bram (0x10000000)


In [42]:
# 构建 VPU Max Pooling 指令序列
print("\n构建指令序列...")

instructions = [
    # Step 1: CDMA 将数据搬到 VPU GB
    encode_cdma_copy(GLOBAL_BRAM_BASE + SRC_ADDR, VPU_GB_BASE + SRC_ADDR, ACT_BYTES),
    encode_wait_cdma(),
    
    # Step 2: 执行 VPU Max Pooling（输入输出都在地址 0）
    encode_vpu_exec(
        unit_choose=UNIT_MP,
        src_addr=SRC_ADDR,  # VPU GB 内部地址 0
        src2_addr=0,
        src_c=SRC_C,
        src_h=SRC_H,
        src_w=SRC_W,
        bias_addr=0,
        scale_addr=0,
        dst_addr=DST_ADDR,  # VPU GB 内部地址 0（覆盖输入）
        addr_break=0,
        addr_s=0,
        addr_t=0
    ),
    encode_wait_vpu(),
    
    # Step 3: CDMA 将结果搬回 global_bram
    encode_cdma_copy(VPU_GB_BASE + DST_ADDR, GLOBAL_BRAM_BASE + DST_ADDR, OUT_BYTES),
    encode_wait_cdma(),
    
    encode_end(),
]

inst_bytes = build_instruction_sequence(instructions)
inst_count = len(inst_bytes) // 4

print(f"✓ 指令序列: {inst_count} words ({len(inst_bytes)} bytes)")
print(f"  CDMA IN:  0x{GLOBAL_BRAM_BASE+SRC_ADDR:08X} → 0x{VPU_GB_BASE+SRC_ADDR:08X} ({ACT_BYTES}B)")
print(f"  VPU:      Max Pooling src=0x{SRC_ADDR:X}, dst=0x{DST_ADDR:X}")
print(f"  CDMA OUT: 0x{VPU_GB_BASE+DST_ADDR:08X} → 0x{GLOBAL_BRAM_BASE+DST_ADDR:08X} ({OUT_BYTES}B)")


构建指令序列...
✓ 指令序列: 25 words (100 bytes)
  CDMA IN:  0x10000000 → 0x10400000 (51200B)
  VPU:      Max Pooling src=0x0, dst=0x0
  CDMA OUT: 0x10400000 → 0x10000000 (51200B)


In [43]:
# 执行 VPU Max Pooling
print("\n执行 VPU 运算...")

# 写入指令
write_blob(INST_BRAM_BASE, inst_bytes)
print("✓ 指令序列已写入 inst_bram")

# 启动解码器
start_time = time.time()
decoder_start(inst_count)
print("✓ 解码器已启动")

# 等待完成
status = decoder_wait(timeout=10.0)
elapsed = time.time() - start_time

print(f"✓ VPU 运算完成: status=0x{status:08X}, 耗时 {elapsed:.3f}s")


执行 VPU 运算...
✓ 指令序列已写入 inst_bram
✓ 解码器已启动
✓ VPU 运算完成: status=0x00000002, 耗时 0.593s


In [44]:
# 读取并验证结果 (FP32)
print("\n读取结果...")

result_bytes = read_blob(GLOBAL_BRAM_BASE + DST_ADDR, OUT_BYTES)
result_fp32 = np.frombuffer(result_bytes, dtype=np.float32)

print(f"✓ 结果数据: {len(result_bytes)} bytes ({len(result_fp32)} FP32)")
print(f"  前8个值: {result_fp32[:8]}")
print(f"  数据范围: [{np.min(result_fp32):.1f}, {np.max(result_fp32):.1f}]")
print(f"  非零元素: {np.count_nonzero(result_fp32)}/{len(result_fp32)}")


读取结果...
✓ 结果数据: 51200 bytes (12800 FP32)
  前8个值: [ 96. 196. 296. 396. 496. 596. 696. 796.]
  数据范围: [92.0, 12799.0]
  非零元素: 12800/12800


In [45]:
# 计算 Golden Model 进行数值比较 (FP32, SAME padding, HWC 格式)
print("\n计算 Golden Model...")

def compute_max_pooling_golden_hwc(input_data_hwc, kernel_size=5, pad=2):
    """
    计算 Max Pooling 的 Golden Model (HWC 格式输入)
    input_data_hwc: shape (H, W, C)
    pad: padding 大小
    返回: shape (H, W, C) (SAME padding时输入输出尺寸相同)
    """
    H, W, C = input_data_hwc.shape
    
    # 应用 padding (用 -inf 填充)
    if pad > 0:
        padded = np.full((H + 2*pad, W + 2*pad, C), -np.inf, dtype=input_data_hwc.dtype)
        padded[pad:pad+H, pad:pad+W, :] = input_data_hwc
    else:
        padded = input_data_hwc
    
    out_h = H  # SAME padding
    out_w = W
    output = np.zeros((out_h, out_w, C), dtype=input_data_hwc.dtype)
    
    for h in range(out_h):
        for w in range(out_w):
            for c in range(C):
                # 提取 kernel 窗口
                window = padded[h:h+kernel_size, w:w+kernel_size, c]
                output[h, w, c] = np.max(window)
    
    return output

# 计算期望结果（输入已经是 HWC 格式）
expected = compute_max_pooling_golden_hwc(act_data, kernel_size=KERNEL_SIZE, pad=PAD)
expected_flat = expected.flatten()

print(f"✓ Golden Model 计算完成")
print(f"  输入形状 (HWC): {act_data.shape}")
print(f"  期望输出形状 (HWC): {expected.shape}")
print(f"  期望数据范围: [{expected.min():.1f}, {expected.max():.1f}]")
print(f"  期望元素数: {expected.size}")
print(f"  实际读取元素数: {len(result_fp32)}")

# 比较结果
if len(result_fp32) >= expected.size:
    result_reshaped = result_fp32[:expected.size].reshape(expected.shape)
    
    diff = np.abs(result_reshaped - expected)
    max_diff = np.max(diff)
    mean_diff = np.mean(diff)
    
    # 计算匹配率
    match_count = np.sum(diff < 0.1)
    match_rate = match_count / expected.size * 100
    
    print(f"\n数值比较:")
    print(f"  最大误差: {max_diff:.4f}")
    print(f"  平均误差: {mean_diff:.4f}")
    print(f"  匹配率: {match_rate:.2f}% ({match_count}/{expected.size})")
    
    # 显示详细对比（前64个值，每行8个）
    print(f"\n详细对比 (前64个值):")
    print("索引 | 期望值  | 实际值  | 误差")
    print("-" * 45)
    num_show = min(64, len(expected_flat))
    for i in range(num_show):
        err = abs(expected_flat[i] - result_fp32[i])
        status = "✓" if err < 0.1 else "✗"
        print(f"{i:3d}  | {expected_flat[i]:7.1f} | {result_fp32[i]:7.1f} | {err:6.2f} {status}")
        if (i + 1) % 16 == 0:
            print()  # 每16个值空一行
    
    # 显示误差分布
    print(f"\n误差分布:")
    print(f"  误差 < 0.01: {np.sum(diff < 0.01)} 个")
    print(f"  误差 < 0.1:  {np.sum(diff < 0.1)} 个")
    print(f"  误差 < 1.0:  {np.sum(diff < 1.0)} 个")
    print(f"  误差 >= 1.0: {np.sum(diff >= 1.0)} 个")
    
    # 找出误差最大的几个位置
    worst_indices = np.argsort(diff.flatten())[-5:][::-1]
    print(f"\n误差最大的5个位置:")
    for idx in worst_indices:
        # HWC 格式: idx = h * (W * C) + w * C + c
        H_out, W_out, C_out = expected.shape
        h = idx // (W_out * C_out)
        w = (idx // C_out) % W_out
        c = idx % C_out
        print(f"  [{idx}] (h={h},w={w},c={c}) 期望={expected_flat[idx]:.2f}, 实际={result_fp32[idx]:.2f}, 误差={diff.flatten()[idx]:.2f}")
    
    if match_rate > 95:
        print(f"\n✓ Max Pooling 数值验证通过!")
    else:
        print(f"\n⚠ Max Pooling 匹配率偏低，需要检查")
else:
    print(f"\n⚠ 数据量不匹配: 期望 {expected.size}, 实际 {len(result_fp32)}")


计算 Golden Model...
✓ Golden Model 计算完成
  输入形状 (HWC): (10, 10, 128)
  期望输出形状 (HWC): (10, 10, 128)
  期望数据范围: [22.0, 12799.0]
  期望元素数: 12800
  实际读取元素数: 12800

数值比较:
  最大误差: 12074.0000
  平均误差: 2205.8938
  匹配率: 0.88% (112/12800)

详细对比 (前64个值):
索引 | 期望值  | 实际值  | 误差
---------------------------------------------
  0  |    22.0 |    96.0 |  74.00 ✗
  1  |   122.0 |   196.0 |  74.00 ✗
  2  |   222.0 |   296.0 |  74.00 ✗
  3  |   322.0 |   396.0 |  74.00 ✗
  4  |   422.0 |   496.0 |  74.00 ✗
  5  |   522.0 |   596.0 |  74.00 ✗
  6  |   622.0 |   696.0 |  74.00 ✗
  7  |   722.0 |   796.0 |  74.00 ✗
  8  |   822.0 |   896.0 |  74.00 ✗
  9  |   922.0 |   996.0 |  74.00 ✗
 10  |  1022.0 |  1096.0 |  74.00 ✗
 11  |  1122.0 |  1196.0 |  74.00 ✗
 12  |  1222.0 |  1296.0 |  74.00 ✗
 13  |  1322.0 |  1396.0 |  74.00 ✗
 14  |  1422.0 |  1496.0 |  74.00 ✗
 15  |  1522.0 |  1596.0 |  74.00 ✗

 16  |  1622.0 |  1696.0 |  74.00 ✗
 17  |  1722.0 |  1796.0 |  74.00 ✗
 18  |  1822.0 |  1896.0 |  74.00 ✗
 19  | 

## 步骤 6: VPU ADD 运算测试

**注意**: ADD 测试当前被跳过，因为：
1. `ad_unit.sv` 可能需要根据新的 BANDWIDTH 参数进行调整
2. 需要先确认 Max Pooling 测试通过后再测试 ADD

如需测试 ADD，请确保：
- `UNIT_AD = 6` (正确的单元代码)
- 数据尺寸是 8 的倍数（FP_CORE_NUM = 8）
- 使用 FP32 数据类型

In [50]:
# 测试 6: VPU ADD 功能（简化配置）
print("=" * 60)
print("测试 6: VPU ADD 运算（最小测试）")
print("=" * 60)

# ★★★ 最小测试：只测试 1 个 BRAM word = 8 个 FP32 ★★★
UNIT_ADD = 6

# 最小数据尺寸（正好 1 个 BRAM word）
ADD_C = 8     # 1 channel
ADD_H = 1     # 1 height  
ADD_W = 1     # 8 width = 8 FP32 = 1 BRAM word
ADD_SIZE = ADD_C * ADD_H * ADD_W  # 8 个 FP32
ADD_BYTES = ADD_SIZE * 4  # 32 bytes = 1 BRAM word

# VPU GB 内地址（字节地址）
ADD_SRC1_ADDR = 0x0000
ADD_SRC2_ADDR = 0x0020  # 32 bytes 后（1 word）
ADD_DST_ADDR = 0x0040   # 64 bytes 后（2 words）

# global_bram 地址
ADD_SRC1_OFF = 0x0
ADD_SRC2_OFF = 0x100
ADD_DST_OFF = 0x200

print(f"ADD 参数 (FP32, 最小测试):")
print(f"  UNIT_CHOOSE: {UNIT_ADD} (UNIT_AD)")
print(f"  FP_CORE_NUM: 8 (每次处理 8 个 FP32)")
print(f"  输入形状: C={ADD_C}, H={ADD_H}, W={ADD_W}")
print(f"  数据大小: {ADD_BYTES} bytes ({ADD_SIZE} FP32) = 1 BRAM word")
print(f"  数据类型: FP32 (float32)")
print(f"  运算: SRC1 + SRC2 = DST")

测试 6: VPU ADD 运算（最小测试）
ADD 参数 (FP32, 最小测试):
  UNIT_CHOOSE: 6 (UNIT_AD)
  FP_CORE_NUM: 8 (每次处理 8 个 FP32)
  输入形状: C=8, H=1, W=1
  数据大小: 32 bytes (8 FP32) = 1 BRAM word
  数据类型: FP32 (float32)
  运算: SRC1 + SRC2 = DST


In [51]:
# 准备ADD测试数据 (FP32, 最小测试)
print("\n准备测试数据 (FP32, 8个元素)...")

# 简单数据：1+10=11, 2+20=22, ..., 8+80=88
src1_data = np.arange(1, 9, dtype=np.float32)  # [1, 2, 3, 4, 5, 6, 7, 8]
src2_data = np.arange(10, 90, 10, dtype=np.float32)  # [10, 20, 30, 40, 50, 60, 70, 80]
expected_add = src1_data + src2_data  # [11, 22, 33, 44, 55, 66, 77, 88]

print(f"✓ SRC1 数据: {src1_data}")
print(f"✓ SRC2 数据: {src2_data}")
print(f"✓ 期望结果: {expected_add}")

# 写入数据到 global_bram
write_blob(GLOBAL_BRAM_BASE + ADD_SRC1_OFF, src1_data.tobytes())
write_blob(GLOBAL_BRAM_BASE + ADD_SRC2_OFF, src2_data.tobytes())

print(f"\n✓ 数据已写入 global_bram")
print(f"  SRC1: 0x{GLOBAL_BRAM_BASE+ADD_SRC1_OFF:08X} ({len(src1_data.tobytes())} bytes)")
print(f"  SRC2: 0x{GLOBAL_BRAM_BASE+ADD_SRC2_OFF:08X} ({len(src2_data.tobytes())} bytes)")

# 预先将 dst_addr 对应范围清零，再执行 add_test
dst_zeros = np.zeros(ADD_SIZE, dtype=np.float32)
dst_zero_bytes = dst_zeros.tobytes()
write_blob(VPU_GB_BASE + ADD_DST_ADDR, dst_zero_bytes)
write_blob(GLOBAL_BRAM_BASE + ADD_DST_OFF, dst_zero_bytes)
print(f"\n✓ DST 区域已预写 0")
print(f"  VPU GB dst: 0x{VPU_GB_BASE+ADD_DST_ADDR:08X} (dst_addr=0x{ADD_DST_ADDR:04X}, {ADD_BYTES} bytes)")
print(f"  global_bram: 0x{GLOBAL_BRAM_BASE+ADD_DST_OFF:08X} ({ADD_BYTES} bytes)")


准备测试数据 (FP32, 8个元素)...
✓ SRC1 数据: [1. 2. 3. 4. 5. 6. 7. 8.]
✓ SRC2 数据: [10. 20. 30. 40. 50. 60. 70. 80.]
✓ 期望结果: [11. 22. 33. 44. 55. 66. 77. 88.]

✓ 数据已写入 global_bram
  SRC1: 0x10000000 (32 bytes)
  SRC2: 0x10000100 (32 bytes)

✓ DST 区域已预写 0
  VPU GB dst: 0x10400040 (dst_addr=0x0040, 32 bytes)
  global_bram: 0x10000200 (32 bytes)


In [52]:
# 构建 VPU ADD 指令序列
print("\n构建指令序列...")

instructions = [
    # Step 1: CDMA 将 SRC1 搬到 VPU GB
    encode_cdma_copy(GLOBAL_BRAM_BASE + ADD_SRC1_OFF, VPU_GB_BASE + ADD_SRC1_ADDR, ADD_BYTES),
    encode_wait_cdma(),
    
    # Step 2: CDMA 将 SRC2 搬到 VPU GB
    encode_cdma_copy(GLOBAL_BRAM_BASE + ADD_SRC2_OFF, VPU_GB_BASE + ADD_SRC2_ADDR, ADD_BYTES),
    encode_wait_cdma(),
    
    # Step 3: 执行 VPU ADD
    encode_vpu_exec(
        unit_choose=UNIT_ADD,
        src_addr=ADD_SRC1_ADDR,  # VPU GB 内部地址
        src2_addr=ADD_SRC2_ADDR,  # 第二个操作数地址
        src_c=ADD_C,
        src_h=ADD_H,
        src_w=ADD_W,
        bias_addr=0,
        scale_addr=0,
        dst_addr=ADD_DST_ADDR,
        addr_break=0,
        addr_s=0,
        addr_t=0
    ),
    encode_wait_vpu(),
    
    # Step 4: CDMA 将结果搬回 global_bram
    encode_cdma_copy(VPU_GB_BASE + ADD_DST_ADDR, GLOBAL_BRAM_BASE + ADD_DST_OFF, ADD_BYTES),
    encode_wait_cdma(),
    
    encode_end(),
]

inst_bytes = build_instruction_sequence(instructions)
inst_count = len(inst_bytes) // 4

print(f"✓ 指令序列: {inst_count} words ({len(inst_bytes)} bytes)")
print(f"  CDMA IN SRC1: 0x{GLOBAL_BRAM_BASE+ADD_SRC1_OFF:08X} → 0x{VPU_GB_BASE+ADD_SRC1_ADDR:08X}")
print(f"  CDMA IN SRC2: 0x{GLOBAL_BRAM_BASE+ADD_SRC2_OFF:08X} → 0x{VPU_GB_BASE+ADD_SRC2_ADDR:08X}")
print(f"  VPU:          ADD (C={ADD_C}, H={ADD_H}, W={ADD_W})")
print(f"  CDMA OUT:     0x{VPU_GB_BASE+ADD_DST_ADDR:08X} → 0x{GLOBAL_BRAM_BASE+ADD_DST_OFF:08X}")


构建指令序列...
✓ 指令序列: 30 words (120 bytes)
  CDMA IN SRC1: 0x10000000 → 0x10400000
  CDMA IN SRC2: 0x10000100 → 0x10400020
  VPU:          ADD (C=8, H=1, W=1)
  CDMA OUT:     0x10400040 → 0x10000200


In [53]:
# 执行 VPU ADD
print("\n执行 VPU 运算...")

# 写入指令
write_blob(INST_BRAM_BASE, inst_bytes)
print("✓ 指令序列已写入 inst_bram")

# 启动解码器
start_time = time.time()
decoder_start(inst_count)
print("✓ 解码器已启动")

# 等待完成
status = decoder_wait(timeout=10.0)
elapsed = time.time() - start_time

print(f"✓ VPU ADD 完成: status=0x{status:08X}, 耗时 {elapsed:.3f}s")


执行 VPU 运算...
✓ 指令序列已写入 inst_bram
✓ 解码器已启动
✓ VPU ADD 完成: status=0x00000002, 耗时 0.609s


In [54]:
# 读取并验证 ADD 结果 (FP32, 最小测试)
print("\n读取结果...")

result_bytes = read_blob(GLOBAL_BRAM_BASE + ADD_DST_OFF, ADD_BYTES)
result_add = np.frombuffer(result_bytes, dtype=np.float32)

print(f"✓ 结果数据: {len(result_bytes)} bytes ({len(result_add)} FP32)")
print(f"  实际输出: {result_add}")
print(f"  期望输出: {expected_add}")

# 数值比较
diff = np.abs(result_add - expected_add)
max_diff = np.max(diff)

# 详细对比
print(f"\n详细对比:")
print("索引 | SRC1  | SRC2  | 期望  | 实际  | 误差  ")
print("-" * 55)
for i in range(len(expected_add)):
    err = diff[i]
    status = "✓" if err < 0.1 else "✗"
    print(f"{i:3d}  | {src1_data[i]:5.1f} | {src2_data[i]:5.1f} | {expected_add[i]:5.1f} | {result_add[i]:5.1f} | {err:5.2f} {status}")

if np.all(diff < 0.1):
    print(f"\n✓ ADD 测试通过！所有 {len(expected_add)} 个元素匹配")
else:
    match_count = np.sum(diff < 0.1)
    print(f"\n✗ ADD 测试失败：{match_count}/{len(expected_add)} 个元素匹配")
    print(f"  最大误差: {max_diff:.2f}")


读取结果...
✓ 结果数据: 32 bytes (8 FP32)
  实际输出: [0. 0. 0. 0. 0. 0. 0. 0.]
  期望输出: [11. 22. 33. 44. 55. 66. 77. 88.]

详细对比:
索引 | SRC1  | SRC2  | 期望  | 实际  | 误差  
-------------------------------------------------------
  0  |   1.0 |  10.0 |  11.0 |   0.0 | 11.00 ✗
  1  |   2.0 |  20.0 |  22.0 |   0.0 | 22.00 ✗
  2  |   3.0 |  30.0 |  33.0 |   0.0 | 33.00 ✗
  3  |   4.0 |  40.0 |  44.0 |   0.0 | 44.00 ✗
  4  |   5.0 |  50.0 |  55.0 |   0.0 | 55.00 ✗
  5  |   6.0 |  60.0 |  66.0 |   0.0 | 66.00 ✗
  6  |   7.0 |  70.0 |  77.0 |   0.0 | 77.00 ✗
  7  |   8.0 |  80.0 |  88.0 |   0.0 | 88.00 ✗

✗ ADD 测试失败：0/8 个元素匹配
  最大误差: 88.00


### ADD 测试问题诊断

如果ADD测试失败，可能的原因：

1. **VPU ADD单元未正确实现** - 需要检查RTL代码中的ADD_Unit模块
2. **地址参数问题** - src_addr 和 src2_addr 可能需要不同的配置
3. **数据布局** - VPU可能期望特定的内存布局
4. **UNIT_CHOOSE值** - ADD对应的单元代码可能不是0

建议调试步骤：
- 检查VPU GB中的数据是否正确写入（上面已验证）
- 查看RTL仿真确认ADD单元是否工作
- 尝试不同的UNIT_CHOOSE值（1, 2, 3等）

## 测试总结

至此，所有VPU硬件测试步骤完成：

1. ✓ BRAM 读写测试 - 验证基础存储访问
2. ✓ VPU 寄存器测试 - 验证配置接口
3. ✓ 指令解码器 - 验证指令执行框架
4. ✓ CDMA 搬运 - 验证数据传输通路
5. ✓ VPU 运算 - 验证 Max Pooling 功能

如果所有步骤都通过，说明VPU硬件工作正常！